<a href="https://colab.research.google.com/github/kartikmeghwal75/CODESOFT/blob/main/myproject2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [11]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kartik2112/fraud-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fraud-detection' dataset.
Path to dataset files: /kaggle/input/fraud-detection


In [12]:
print(df.shape)

print(df.info())

print(df.isnull().sum())

(555719, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 19 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   merchant    555719 non-null  object 
 1   category    555719 non-null  object 
 2   amt         555719 non-null  float64
 3   gender      555719 non-null  object 
 4   state       555719 non-null  object 
 5   zip         555719 non-null  int64  
 6   lat         555719 non-null  float64
 7   long        555719 non-null  float64
 8   city_pop    555719 non-null  int64  
 9   job         555719 non-null  object 
 10  dob         555719 non-null  object 
 11  unix_time   555719 non-null  int64  
 12  merch_lat   555719 non-null  float64
 13  merch_long  555719 non-null  float64
 14  is_fraud    555719 non-null  int64  
 15  year        555719 non-null  int32  
 16  month       555719 non-null  int32  
 17  day         555719 non-null  int32  
 18  hour        555719 non-null  in

In [13]:
print(df["is_fraud"].value_counts())

is_fraud
0    553574
1      2145
Name: count, dtype: int64


In [18]:
df["dob"] = pd.to_datetime(df["dob"])

df["age"] = 2020 - df["dob"].dt.year

df.drop("dob", axis=1, inplace=True)

In [19]:
X = df.drop("is_fraud", axis=1)

y = df["is_fraud"]

In [20]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns

cat_cols = X.select_dtypes(include=["object"]).columns

In [21]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced"))
])

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, lr_pred))

print(classification_report(y_test, lr_pred))

print(confusion_matrix(y_test, lr_pred))

print("ROC AUC:",
      roc_auc_score(
          y_test,
          lr_model.predict_proba(X_test)[:,1]
      ))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.9093338371841935
              precision    recall  f1-score   support

           0       1.00      0.91      0.95    110715
           1       0.03      0.82      0.07       429

    accuracy                           0.91    111144
   macro avg       0.52      0.87      0.51    111144
weighted avg       1.00      0.91      0.95    111144

[[100714  10001]
 [    76    353]]
ROC AUC: 0.9505974884378893


In [24]:
dt_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, dt_pred))

print(classification_report(y_test, dt_pred))

print(confusion_matrix(y_test, dt_pred))

print("ROC AUC:",
      roc_auc_score(
          y_test,
          dt_model.predict_proba(X_test)[:,1]
      ))

Accuracy: 0.9967429640826315
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    110715
           1       0.56      0.72      0.63       429

    accuracy                           1.00    111144
   macro avg       0.78      0.86      0.81    111144
weighted avg       1.00      1.00      1.00    111144

[[110472    243]
 [   119    310]]
ROC AUC: 0.8602079490306017


In [25]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))

print(confusion_matrix(y_test, rf_pred))

print("ROC AUC:",
      roc_auc_score(
          y_test,
          rf_model.predict_proba(X_test)[:,1]
      ))

Accuracy: 0.9980115885697833
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    110715
           1       0.96      0.51      0.66       429

    accuracy                           1.00    111144
   macro avg       0.98      0.75      0.83    111144
weighted avg       1.00      1.00      1.00    111144

[[110706      9]
 [   212    217]]
ROC AUC: 0.988300648455099


In [26]:
models = {
    "Logistic Regression": lr_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model
}

for name, model in models.items():

    pred = model.predict(X_test)

    prob = model.predict_proba(X_test)[:,1]

    print("="*40)
    print(name)

    print("Accuracy:",
          accuracy_score(y_test, pred))

    print("ROC AUC:",
          roc_auc_score(y_test, prob))

Logistic Regression
Accuracy: 0.9093338371841935
ROC AUC: 0.9505974884378893
Decision Tree
Accuracy: 0.9967429640826315
ROC AUC: 0.8602079490306017
Random Forest
Accuracy: 0.9980115885697833
ROC AUC: 0.988300648455099


In [27]:
sample = X_test.iloc[[0]]

prediction = rf_model.predict(sample)

if prediction[0] == 1:
    print("Fraudulent Transaction")
else:
    print("Legitimate Transaction")

Legitimate Transaction
